# Learning drag on random radial closed polylines

This notebook builds an online training pipeline: it samples a radial closed polyline, estimates its two-dimensional viscous drag in left-to-right flow with a batched D2Q9 lattice-Boltzmann (LBM) solver, and trains a Transformer to regress the resulting drag coefficient. The polyline is represented by `(sin(theta), cos(theta), radius)`, where `theta` is measured from the positive vertical axis.

The final edge from the last vertex back to the first is implicit. Angles follow their cyclic order around the circle. A random positive normalized Fourier kernel smooths the radii by circular convolution, and every polygon is normalized to the same area.

In [ ]:
from dataclasses import dataclass
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display
from torch import nn
import torch.nn.functional as F
from tqdm.auto import tqdm, trange

torch.manual_seed(7)
np.random.seed(7)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}; device={DEVICE}")


@dataclass(frozen=True)
class FlowConfig:
    # The physical cell sizes are nearly square. Increase nx/ny and steps for
    # higher-fidelity labels; these defaults are intended for online training.
    nx: int = 96
    ny: int = 64
    xlim: tuple[float, float] = (-3.0, 5.0)
    ylim: tuple[float, float] = (-8.0 / 3.0, 8.0 / 3.0)
    inlet_speed: float = 0.055       # lattice units / step
    viscosity: float = 0.025         # lattice kinematic viscosity
    steps: int = 320
    average_last: int = 100
    boundary_width: int = 2


FLOW = FlowConfig()
OBJECT_MIN_RADIUS = 0.15
OBJECT_MAX_RADIUS = 0.65  # smaller than the previous radius-1 obstacles
OBJECT_TARGET_AREA = 0.35  # equivalent circle radius is about 0.33
MIN_RADIAL_CV, MAX_RADIAL_CV = 0.16, 0.34
MIN_CIRCULARITY, MAX_CIRCULARITY = 0.45, 0.82
MIN_KERNEL_ENTROPY, MAX_KERNEL_ENTROPY = 0.25, 0.65
ELONGATED_SHAPE_PROBABILITY = 0.55
MIN_ELONGATED_ASPECT_RATIO = 1.50
dx = (FLOW.xlim[1] - FLOW.xlim[0]) / (FLOW.nx - 1)
maximum_reynolds = FLOW.inlet_speed * (2.0 * OBJECT_MAX_RADIUS / dx) / FLOW.viscosity
print(f"Grid={FLOW.nx}x{FLOW.ny}; maximum-radius Re = {maximum_reynolds:.1f}")

## Random radial closed polylines

Angles form a jittered cyclic partition of the circle and vertices are connected without permuting that order. Each shape samples random Fourier coefficients, inverse-transforms them into a circular kernel, and adjusts its softmax temperature to target normalized kernel entropy in `[0.25, 0.65]`. Circular convolution is combined with a dominant harmonic and normalized to radial coefficient of variation in `[0.16, 0.34]`. About 55% of samples deliberately use a strong second harmonic and must reach principal-axis aspect ratio `1.5`, producing a substantial elongated family. Candidates outside circularity `[0.45, 0.82]` are resampled, strongly excluding near-circles. Finally, all radii are scaled so the straight-edged polygon has area `0.35` (equivalent circle radius about `0.33`). The maximum permitted radius remains `0.65`.

In [ ]:
def sample_fourier_smoothing_kernel(
    point_count: int, device: torch.device
) -> torch.Tensor:
    """Sample a positive circular kernel with controlled entropy."""
    mode_count = point_count // 2 + 1
    frequency = torch.arange(mode_count, device=device, dtype=torch.float32)
    spectral_decay = (1.0 + frequency).pow(-0.75)
    real_coefficients = (
        math.sqrt(point_count)
        * spectral_decay
        * torch.randn(mode_count, device=device)
    )
    real_coefficients[0] = 0.0  # softmax is invariant to a DC offset
    spectrum = torch.complex(real_coefficients, torch.zeros_like(real_coefficients))
    logits = torch.fft.irfft(spectrum, n=point_count, norm="ortho")
    target_entropy = MIN_KERNEL_ENTROPY + (
        MAX_KERNEL_ENTROPY - MIN_KERNEL_ENTROPY
    ) * torch.rand((), device=device)

    # Normalized entropy rises monotonically with softmax temperature.
    lower_temperature = torch.tensor(1e-3, device=device)
    upper_temperature = torch.tensor(100.0, device=device)
    for _ in range(16):
        temperature = 0.5 * (lower_temperature + upper_temperature)
        trial_kernel = torch.softmax(logits / temperature, dim=0)
        entropy = -(
            trial_kernel * trial_kernel.clamp_min(1e-12).log()
        ).sum() / math.log(point_count)
        if entropy < target_entropy:
            lower_temperature = temperature
        else:
            upper_temperature = temperature
    return torch.softmax(logits / upper_temperature, dim=0)


def sample_random_polylines(
    batch_size: int,
    min_points: int = 5,
    max_points: int = 50,
    min_radius: float = OBJECT_MIN_RADIUS,
    max_radius: float = OBJECT_MAX_RADIUS,
    device: torch.device = DEVICE,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Return radial sequences, lengths, and Fourier smoothing kernels.

    theta is measured from +vertical, hence x=r*sin(theta), y=r*cos(theta).
    Vertices retain cyclic angular order and every polygon has constant area.
    CUDA targets are sampled on CPU and transferred once; this avoids hundreds
    of tiny synchronized GPU launches without changing the distribution.
    """
    if not (3 <= min_points <= max_points):
        raise ValueError("Expected 3 <= min_points <= max_points")
    if not (0.0 < min_radius < max_radius):
        raise ValueError("Expected 0 < min_radius < max_radius")

    target_device = torch.device(device)
    device = (
        torch.device("cpu")
        if target_device.type == "cuda"
        else target_device
    )
    lengths = torch.randint(min_points, max_points + 1, (batch_size,), device=device)
    parameters = torch.zeros(batch_size, max_points, 3, device=device)
    smoothing_kernels = torch.zeros(batch_size, max_points, device=device)

    for b, n_tensor in enumerate(lengths):
        n = int(n_tensor)
        make_elongated = bool(
            torch.rand((), device=device) < ELONGATED_SHAPE_PROBABILITY
        )
        accepted = False
        for _ in range(96):
            phase = 2.0 * math.pi * torch.rand((), device=device)
            jitter = (
                (torch.rand(n, device=device) - 0.5)
                * (0.70 * 2.0 * math.pi / n)
            )
            theta = torch.remainder(
                phase
                + 2.0 * math.pi * torch.arange(n, device=device) / n
                + jitter,
                2.0 * math.pi,
            )
            angular_gap = torch.remainder(
                torch.roll(theta, -1) - theta, 2.0 * math.pi
            )

            raw_radius = min_radius + (max_radius - min_radius) * torch.rand(
                n, device=device
            )
            kernel = sample_fourier_smoothing_kernel(n, device)
            convolved = torch.fft.irfft(
                torch.fft.rfft(raw_radius) * torch.fft.rfft(kernel), n=n
            )
            convolved = convolved - convolved.mean()
            convolved = convolved / convolved.std(unbiased=False).clamp_min(1e-6)

            maximum_mode = min(6, max(1, (n - 1) // 2))
            if make_elongated:
                dominant_mode = torch.tensor(2, device=device)
                dominant_weight = 0.65 + 0.20 * torch.rand((), device=device)
            else:
                dominant_mode = torch.randint(
                    1, maximum_mode + 1, (), device=device
                )
                dominant_weight = 0.30 + 0.40 * torch.rand((), device=device)
            dominant_phase = 2.0 * math.pi * torch.rand((), device=device)
            dominant = torch.cos(dominant_mode * theta + dominant_phase)
            dominant = dominant - dominant.mean()
            dominant = dominant / dominant.std(unbiased=False).clamp_min(1e-6)
            variation = (
                (1.0 - dominant_weight) * convolved
                + dominant_weight * dominant
            )
            variation = variation - variation.mean()
            variation = variation / variation.std(unbiased=False).clamp_min(1e-6)

            minimum_target_cv = 0.22 if make_elongated else MIN_RADIAL_CV
            target_cv = minimum_target_cv + (
                MAX_RADIAL_CV - minimum_target_cv
            ) * torch.rand((), device=device)
            lower_amplitude = torch.zeros((), device=device)
            upper_amplitude = torch.tensor(2.0, device=device)
            for _ in range(12):
                amplitude = 0.5 * (lower_amplitude + upper_amplitude)
                trial_profile = torch.exp(amplitude * variation)
                trial_cv = (
                    trial_profile.std(unbiased=False)
                    / trial_profile.mean().clamp_min(1e-8)
                )
                if trial_cv < target_cv:
                    lower_amplitude = amplitude
                else:
                    upper_amplitude = amplitude
            profile = torch.exp(
                0.5 * (lower_amplitude + upper_amplitude) * variation
            )
            profile_area = 0.5 * torch.sum(
                profile * torch.roll(profile, -1) * torch.sin(angular_gap)
            )
            radius = profile * torch.sqrt(
                OBJECT_TARGET_AREA / profile_area.clamp_min(1e-8)
            )

            edge_length = torch.sqrt(
                radius.square()
                + torch.roll(radius, -1).square()
                - 2.0
                * radius
                * torch.roll(radius, -1)
                * torch.cos(angular_gap)
            ).clamp_min(1e-8)
            perimeter = edge_length.sum()
            circularity = 4.0 * math.pi * OBJECT_TARGET_AREA / perimeter.square()
            radial_cv = radius.std(unbiased=False) / radius.mean().clamp_min(1e-8)
            vertices = torch.stack(
                (radius * torch.sin(theta), radius * torch.cos(theta)), dim=-1
            )
            centered_vertices = vertices - vertices.mean(dim=0)
            covariance = centered_vertices.T @ centered_vertices / n
            eigenvalues = torch.linalg.eigvalsh(covariance).clamp_min(1e-8)
            aspect_ratio = torch.sqrt(eigenvalues[-1] / eigenvalues[0])
            within_bounds = radius.min() >= min_radius and radius.max() <= max_radius
            sufficiently_elongated = (
                not make_elongated
                or aspect_ratio >= MIN_ELONGATED_ASPECT_RATIO
            )
            diverse = (
                radial_cv >= MIN_RADIAL_CV * 0.98
                and radial_cv <= MAX_RADIAL_CV * 1.02
                and circularity >= MIN_CIRCULARITY
                and circularity <= MAX_CIRCULARITY
            )
            if within_bounds and diverse and sufficiently_elongated:
                accepted = True
                break

        if not accepted:
            raise RuntimeError(
                f"Could not sample a diverse {n}-point shape within 96 attempts"
            )
        smoothing_kernels[b, :n] = kernel
        parameters[b, :n] = torch.stack(
            (torch.sin(theta), torch.cos(theta), radius), dim=-1
        )

    return (
        parameters.to(target_device),
        lengths.to(target_device),
        smoothing_kernels.to(target_device),
    )


def parameters_to_vertices(parameters: torch.Tensor) -> torch.Tensor:
    """Convert (..., 3) parameter vectors to (..., 2) Cartesian vertices."""
    radius = parameters[..., 2]
    return torch.stack((radius * parameters[..., 0], radius * parameters[..., 1]), dim=-1)


def polyline_areas(parameters: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    """Exact shoelace area of each valid closed polyline."""
    vertices = parameters_to_vertices(parameters)
    batch_size, sequence_length, _ = vertices.shape
    index = torch.arange(sequence_length, device=parameters.device)[None, :]
    valid = index < lengths[:, None]
    next_index = torch.remainder(index + 1, lengths[:, None]).expand(batch_size, -1)
    next_vertices = vertices.gather(
        1, next_index[..., None].expand(-1, -1, 2)
    )
    cross = (
        vertices[..., 0] * next_vertices[..., 1]
        - vertices[..., 1] * next_vertices[..., 0]
    )
    return 0.5 * torch.abs((cross * valid).sum(dim=1))


## Batched LBM drag simulation

The solver uses a D2Q9 BGK collision operator. Constant-velocity equilibrium populations are imposed in a two-cell far-field frame, providing left-to-right flow without solid channel walls. Halfway bounce-back enforces no-slip on every non-zero-winding interior of the rasterized polyline. The force is measured directly from momentum exchanged on fluid-solid links and averaged near the end of the run.

The returned target is the two-dimensional drag coefficient `Cd = Fx / (0.5 * rho * U^2 * D)`, with the filled shape's vertical span as reference length. All geometries in a minibatch are advanced together.

In [ ]:
def rasterize_polylines(
    parameters: torch.Tensor, lengths: torch.Tensor, config: FlowConfig = FLOW
) -> torch.Tensor:
    """Fill every non-zero-winding interior of a padded polyline batch."""
    batch_size, max_vertices, _ = parameters.shape
    device = parameters.device
    vertices = parameters_to_vertices(parameters)

    xs = torch.linspace(*config.xlim, config.nx, device=device)
    ys = torch.linspace(*config.ylim, config.ny, device=device)
    grid_y, grid_x = torch.meshgrid(ys, xs, indexing="ij")
    winding_number = torch.zeros(
        batch_size, config.ny, config.nx, dtype=torch.int16, device=device
    )

    edge_index = torch.arange(max_vertices, device=device)[None, :].expand(batch_size, -1)
    next_index = torch.remainder(edge_index + 1, lengths[:, None])
    next_vertices = vertices.gather(1, next_index[..., None].expand(-1, -1, 2))

    for edge in range(max_vertices):
        valid = edge < lengths
        x0 = vertices[:, edge, 0, None, None]
        y0 = vertices[:, edge, 1, None, None]
        x1 = next_vertices[:, edge, 0, None, None]
        y1 = next_vertices[:, edge, 1, None, None]
        is_left = (x1 - x0) * (grid_y - y0) - (grid_x - x0) * (y1 - y0)
        upward_crossing = (y0 <= grid_y) & (y1 > grid_y) & (is_left > 0.0)
        downward_crossing = (y0 > grid_y) & (y1 <= grid_y) & (is_left < 0.0)
        winding_number += valid[:, None, None] * (
            upward_crossing.to(torch.int16) - downward_crossing.to(torch.int16)
        )

    return winding_number != 0


def _equilibrium(
    rho: torch.Tensor, ux: torch.Tensor, uy: torch.Tensor,
    cx: torch.Tensor, cy: torch.Tensor, weights: torch.Tensor,
) -> torch.Tensor:
    cu = ux[:, None] * cx[None, :, None, None] + uy[:, None] * cy[None, :, None, None]
    speed2 = ux.square() + uy.square()
    return weights[None, :, None, None] * rho[:, None] * (
        1.0 + 3.0 * cu + 4.5 * cu.square() - 1.5 * speed2[:, None]
    )


D2Q9_DIRECTIONS = (
    (0, 0), (1, 0), (0, 1), (-1, 0), (0, -1),
    (1, 1), (-1, 1), (-1, -1), (1, -1),
)
D2Q9_OPPOSITE = (0, 3, 4, 1, 2, 7, 8, 5, 6)


def _lbm_step(
    populations: torch.Tensor,
    solid: torch.Tensor,
    fluid: torch.Tensor,
    hit: torch.Tensor,
    far_equilibrium: torch.Tensor,
    far_field: torch.Tensor,
    cx: torch.Tensor,
    cy: torch.Tensor,
    weights: torch.Tensor,
    omega: float,
) -> tuple[torch.Tensor, torch.Tensor]:
    """One physically identical LBM step, structured for compilation."""
    rho = populations.sum(dim=1).clamp_min(1e-8)
    ux = (populations * cx[None, :, None, None]).sum(dim=1) / rho
    uy = (populations * cy[None, :, None, None]).sum(dim=1) / rho
    ux = ux.masked_fill(solid, 0.0)
    uy = uy.masked_fill(solid, 0.0)
    equilibrium = _equilibrium(rho, ux, uy, cx, cy, weights)
    post_collision = (
        populations - omega * (populations - equilibrium)
    ) * fluid[:, None]

    streamed = torch.zeros_like(populations)
    force_x = torch.zeros(
        populations.shape[0],
        device=populations.device,
        dtype=populations.dtype,
    )
    for direction, (shift_x, shift_y) in enumerate(D2Q9_DIRECTIONS):
        moving = post_collision[:, direction]
        bounced = moving * hit[:, direction]
        streamed[:, D2Q9_OPPOSITE[direction]] += bounced
        streamed[:, direction] += torch.roll(
            moving.masked_fill(hit[:, direction], 0.0),
            shifts=(shift_y, shift_x),
            dims=(-2, -1),
        )
        force_x += 2.0 * shift_x * bounced.sum(dim=(-2, -1))

    streamed = streamed.masked_fill(solid[:, None], 0.0)
    streamed = torch.where(
        far_field[None, None], far_equilibrium, streamed
    )
    return streamed, force_x


# Compilation changes execution only; equations and FP32 precision are
# unchanged. A new batch shape pays a one-time compilation cost.
_advance_lbm_step = (
    torch.compile(_lbm_step, mode="default", fullgraph=True)
    if DEVICE.type == "cuda"
    else _lbm_step
)


@torch.inference_mode()
def estimate_drag_lbm(
    parameters: torch.Tensor,
    lengths: torch.Tensor,
    config: FlowConfig = FLOW,
    return_mask: bool = False,
) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
    """Estimate Cd using a compiled sharp-boundary momentum-exchange LBM."""
    device = parameters.device
    dtype = torch.float32
    parameters = parameters.to(dtype)
    batch_size = len(parameters)
    solid = rasterize_polylines(parameters, lengths, config)
    if torch.any(solid.sum(dim=(-2, -1)) == 0):
        raise ValueError(
            "A sampled polyline has no resolved interior; "
            "increase its radius or grid resolution"
        )
    fluid = ~solid

    directions = torch.tensor(
        D2Q9_DIRECTIONS, device=device, dtype=dtype
    )
    cx, cy = directions[:, 0], directions[:, 1]
    weights = torch.tensor(
        [4 / 9, 1 / 9, 1 / 9, 1 / 9, 1 / 9,
         1 / 36, 1 / 36, 1 / 36, 1 / 36],
        device=device,
        dtype=dtype,
    )
    omega = 1.0 / (0.5 + 3.0 * config.viscosity)

    rho0 = torch.ones(
        batch_size, config.ny, config.nx, device=device, dtype=dtype
    )
    ux0 = torch.full_like(rho0, config.inlet_speed)
    uy0 = torch.zeros_like(rho0)
    far_equilibrium = _equilibrium(rho0, ux0, uy0, cx, cy, weights)
    populations = far_equilibrium.clone()
    populations.masked_fill_(solid[:, None], 0.0)

    width = config.boundary_width
    far_field = torch.zeros(
        config.ny, config.nx, dtype=torch.bool, device=device
    )
    far_field[:width] = True
    far_field[-width:] = True
    far_field[:, :width] = True
    far_field[:, -width:] = True
    if torch.any(solid & far_field):
        raise ValueError("An obstacle touches the far-field boundary")

    hit = torch.zeros(
        batch_size, 9, config.ny, config.nx,
        dtype=torch.bool, device=device,
    )
    for direction, (shift_x, shift_y) in enumerate(
        D2Q9_DIRECTIONS[1:], start=1
    ):
        destination_is_solid = torch.roll(
            solid, shifts=(-shift_y, -shift_x), dims=(-2, -1)
        )
        hit[:, direction] = fluid & destination_is_solid

    force_sum = torch.zeros(batch_size, device=device, dtype=dtype)
    sample_from = max(0, config.steps - config.average_last)
    for step in range(config.steps):
        populations, force_x = _advance_lbm_step(
            populations, solid, fluid, hit, far_equilibrium, far_field,
            cx, cy, weights, omega,
        )
        if step >= sample_from:
            force_sum += force_x

    mean_force_x = force_sum / float(config.average_last)
    occupied_row = solid.any(dim=-1)
    row_index = torch.arange(
        config.ny, device=device
    )[None, :].expand(batch_size, -1)
    top_row = row_index.masked_fill(~occupied_row, -1).max(dim=1).values
    bottom_row = row_index.masked_fill(
        ~occupied_row, config.ny
    ).min(dim=1).values
    reference_length_lattice = (
        top_row - bottom_row + 1
    ).clamp_min(1).to(dtype)
    cd = mean_force_x / (
        0.5 * config.inlet_speed**2 * reference_length_lattice
    )
    cd = cd.clamp_min(0.0)
    return (cd, solid) if return_mask else cd


# Compile the batch size used by training and verify the optimized path.
smoke_parameters, smoke_lengths, _ = sample_random_polylines(16)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
start = time.perf_counter()
smoke_drag, smoke_masks = estimate_drag_lbm(
    smoke_parameters, smoke_lengths, return_mask=True
)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
elapsed = time.perf_counter() - start
print(
    f"Cd(first four)={smoke_drag[:4].cpu().numpy().round(3)} "
    f"({elapsed:.2f}s for 16 shapes; first call includes compilation)"
)
assert torch.isfinite(smoke_drag).all() and (smoke_drag > 0).all()


## Drag-coefficient density

Simulate 512 fresh shapes in memory-bounded batches and plot the density histogram of their drag coefficients.

In [ ]:
@torch.inference_mode()
def sample_cd_distribution(
    sample_count: int = 512,
    simulation_batch_size: int | None = None,
    flow_config: FlowConfig = FLOW,
) -> torch.Tensor:
    if simulation_batch_size is None:
        simulation_batch_size = 32 if DEVICE.type == "cuda" else 4

    drag_batches = []
    progress = tqdm(total=sample_count, desc="sampling Cd", unit="shape")
    for start in range(0, sample_count, simulation_batch_size):
        count = min(simulation_batch_size, sample_count - start)
        parameters, lengths, _ = sample_random_polylines(count, device=DEVICE)
        drag_batches.append(estimate_drag_lbm(parameters, lengths, flow_config).cpu())
        progress.update(count)
    progress.close()
    return torch.cat(drag_batches)


cd_samples = sample_cd_distribution(sample_count=512)
cd_mean = cd_samples.mean().item()
cd_median = cd_samples.median().item()
cd_std = cd_samples.std().item()
CD_NORMALIZATION_MEAN = cd_mean
CD_NORMALIZATION_STD = max(cd_std, 1e-6)
normalized_cd_samples = (
    cd_samples - CD_NORMALIZATION_MEAN
) / CD_NORMALIZATION_STD

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(
    cd_samples.numpy(), bins=30, density=True,
    color="tab:blue", alpha=0.72, edgecolor="white", linewidth=0.7,
)
ax.axvline(cd_mean, color="tab:red", lw=2, label=f"mean = {cd_mean:.3f}")
ax.axvline(
    cd_median, color="tab:orange", lw=2, ls="--",
    label=f"median = {cd_median:.3f}",
)
ax.text(
    0.98, 0.95,
    f"n = {len(cd_samples)}\nstd = {cd_std:.3f}\n"
    f"range = [{cd_samples.min():.3f}, {cd_samples.max():.3f}]",
    transform=ax.transAxes, ha="right", va="top",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.9, edgecolor="0.8"),
)
ax.set(
    xlabel=r"drag coefficient $C_d$", ylabel="probability density",
    title="Drag-coefficient density for 512 random radial polylines",
)
normalized_axis = ax.secondary_xaxis(
    "top",
    functions=(
        lambda drag: (drag - CD_NORMALIZATION_MEAN) / CD_NORMALIZATION_STD,
        lambda normalized: normalized * CD_NORMALIZATION_STD + CD_NORMALIZATION_MEAN,
    ),
)
normalized_axis.set_xlabel(r"normalized drag $z_{C_d}$")
ax.grid(axis="y", alpha=0.2)
ax.legend()
plt.show()

print(
    f"Cd mean={cd_mean:.4f}, median={cd_median:.4f}, std={cd_std:.4f}, "
    f"min={cd_samples.min().item():.4f}, max={cd_samples.max().item():.4f}"
)

## Geometry-Fourier Transformer

Each vertex's `sin(theta)`, `cos(theta)`, and radius are expanded independently with fixed multiscale Fourier features before projection to dimension 256. No index-based positional encoding is used: absolute angular geometry is already present in the vertex features, and the star-convex shape is determined by the angular set of radii. Padding is masked in both attention and pooling. The regressor predicts standardized drag, `z = (Cd - mean(Cd)) / std(Cd)`, using the 512-sample calibration distribution above.

In [ ]:
class GeometryFourierFeatures(nn.Module):
    def __init__(self, bands: int = 32, max_frequency: float = 16.0):
        super().__init__()
        if bands < 1 or max_frequency < 1.0:
            raise ValueError("Expected bands >= 1 and max_frequency >= 1")
        frequencies = math.pi * 2.0 ** torch.linspace(
            0.0, math.log2(max_frequency), bands
        )
        self.register_buffer("frequencies", frequencies)
        self.output_dimension = 3 * (1 + 2 * bands)

    def forward(self, parameters: torch.Tensor) -> torch.Tensor:
        phases = parameters[..., None] * self.frequencies
        features = torch.cat(
            (parameters[..., None], phases.sin(), phases.cos()), dim=-1
        )
        return features.flatten(-2)


class DragTransformer(nn.Module):
    def __init__(
        self,
        dimension: int = 128,
        heads: int = 4,
        layers: int = 4,
        dropout: float = 0.0,
        fourier_bands: int = 32,
        max_fourier_frequency: float = 16.0,
    ):
        super().__init__()
        self.fourier_features = GeometryFourierFeatures(
            bands=fourier_bands, max_frequency=max_fourier_frequency
        )
        self.input_projection = nn.Sequential(
            nn.Linear(self.fourier_features.output_dimension, dimension),
            nn.LayerNorm(dimension),
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dimension,
            nhead=heads,
            dim_feedforward=4 * dimension,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=layers, norm=nn.LayerNorm(dimension),
            enable_nested_tensor=False,
        )
        self.regressor = nn.Sequential(
            nn.Linear(dimension, dimension),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dimension, 1),
        )

    def forward(self, parameters: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        sequence_length = parameters.shape[1]
        index = torch.arange(sequence_length, device=parameters.device)[None, :]
        padding_mask = index >= lengths[:, None]
        hidden = self.input_projection(self.fourier_features(parameters))
        hidden = self.encoder(hidden, src_key_padding_mask=padding_mask)
        valid = (~padding_mask)[..., None]
        pooled = (hidden * valid).sum(dim=1) / lengths[:, None]
        # Standardized drag may be negative, so the output is unconstrained.
        return self.regressor(pooled).squeeze(-1)


model = DragTransformer(dimension=256, heads=8, layers=8).to(DEVICE)
with torch.inference_mode():
    smoke_prediction = model(smoke_parameters, smoke_lengths)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Model parameters: {parameter_count / 1e6:.2f}M; output shape={tuple(smoke_prediction.shape)}")

## Online training and 100-step visual checks

Every training item is newly sampled and simulated; labels are not recycled from a fixed dataset. At the beginning of training, 40 fresh Fourier-smoothed shapes are rendered as a generator debug preview. The loss is Huber loss between predicted and true standardized drag coefficients. Every 100 optimizer steps, ten fresh held-out shapes are simulated and rendered with physical true and predicted drag.

The projected-gradient-descent implementation is retained but temporarily disabled by default with `TrainingConfig.adversarial_enabled=False`. Set it to `True` to restore bounded angle/radius attacks, adversarial loss, PGD validation, and PGD plot overlays.

In [ ]:
@dataclass(frozen=True)
class TrainingConfig:
    steps: int = 100_000
    batch_size: int = 16 if torch.cuda.is_available() else 4
    learning_rate: float = 3e-5
    weight_decay: float = 1e-4
    plot_every: int = 200
    validation_shapes: int = 10
    gradient_clip: float = 1.0
    adversarial_enabled: bool = False  # temporarily disabled
    adversarial_weight: float = 0.50
    attack_steps: int = 3
    attack_angle_epsilon: float = 0.012  # radians
    attack_radius_epsilon: float = 0.03
    attack_step_fraction: float = 0.50


def normalize_drag(drag: torch.Tensor) -> torch.Tensor:
    return (drag - CD_NORMALIZATION_MEAN) / CD_NORMALIZATION_STD


def denormalize_drag(normalized_drag: torch.Tensor) -> torch.Tensor:
    return normalized_drag * CD_NORMALIZATION_STD + CD_NORMALIZATION_MEAN


def drag_regression_loss(
    normalized_predictions: torch.Tensor, targets: torch.Tensor
) -> torch.Tensor:
    return F.smooth_l1_loss(
        normalized_predictions, normalize_drag(targets), beta=0.10
    )


def pgd_adversarial_parameters(
    model: nn.Module,
    parameters: torch.Tensor,
    lengths: torch.Tensor,
    targets: torch.Tensor,
    config: TrainingConfig,
    random_start: bool = True,
) -> torch.Tensor:
    """Maximize regression loss with a bounded PGD parameter attack."""
    was_training = model.training
    model.eval()  # deterministic gradients: disable Transformer dropout during PGD
    sequence_length = parameters.shape[1]
    index = torch.arange(sequence_length, device=parameters.device)[None, :]
    valid = index < lengths[:, None]
    theta = torch.remainder(
        torch.atan2(parameters[..., 0], parameters[..., 1]), 2.0 * math.pi
    ).detach()
    radius = parameters[..., 2].detach()

    if random_start:
        delta_theta = (
            (2.0 * torch.rand_like(theta) - 1.0) * config.attack_angle_epsilon
        )
        delta_radius = (2.0 * torch.rand_like(radius) - 1.0) * config.attack_radius_epsilon
    else:
        delta_theta = torch.zeros_like(theta)
        delta_radius = torch.zeros_like(radius)
    delta_theta *= valid
    delta_radius *= valid

    def project(delta_angle: torch.Tensor, delta_r: torch.Tensor) -> torch.Tensor:
        attacked_theta = theta + delta_angle
        attacked_radius = (radius + delta_r).clamp(
            OBJECT_MIN_RADIUS, OBJECT_MAX_RADIUS
        )
        attacked = torch.stack(
            (attacked_theta.sin(), attacked_theta.cos(), attacked_radius), dim=-1
        )
        return attacked * valid[..., None]

    angle_step = config.attack_step_fraction * config.attack_angle_epsilon
    radius_step = config.attack_step_fraction * config.attack_radius_epsilon
    for _ in range(config.attack_steps):
        delta_theta.requires_grad_(True)
        delta_radius.requires_grad_(True)
        attacked = project(delta_theta, delta_radius)
        attack_loss = drag_regression_loss(model(attacked, lengths), targets)
        angle_gradient, radius_gradient = torch.autograd.grad(
            attack_loss, (delta_theta, delta_radius), only_inputs=True
        )
        with torch.no_grad():
            delta_theta = delta_theta + angle_step * angle_gradient.sign()
            delta_theta = delta_theta.clamp(
                -config.attack_angle_epsilon, config.attack_angle_epsilon
            )
            delta_radius = (delta_radius + radius_step * radius_gradient.sign()).clamp(
                -config.attack_radius_epsilon, config.attack_radius_epsilon
            )
            delta_theta *= valid
            delta_radius *= valid

    attacked = project(delta_theta.detach(), delta_radius.detach()).detach()
    model.train(was_training)
    return attacked


def render_sampled_shapes(
    parameters: torch.Tensor, lengths: torch.Tensor
) -> None:
    """Debug preview shown once, immediately before training starts."""
    parameters = parameters.detach().cpu()
    lengths = lengths.detach().cpu()
    fig, axes = plt.subplots(5, 8, figsize=(15, 9), constrained_layout=True)
    for ax, param, length in zip(axes.flat, parameters, lengths):
        vertices = parameters_to_vertices(param[:length]).numpy()
        closed = np.vstack((vertices, vertices[0]))
        radii = param[:length, 2].numpy()
        radial_cv = radii.std() / max(radii.mean(), 1e-8)
        perimeter = np.linalg.norm(np.roll(vertices, -1, axis=0) - vertices, axis=1).sum()
        circularity = 4.0 * math.pi * OBJECT_TARGET_AREA / max(perimeter**2, 1e-8)
        centered_vertices = vertices - vertices.mean(axis=0)
        eigenvalues = np.linalg.eigvalsh(
            centered_vertices.T @ centered_vertices / len(vertices)
        )
        aspect_ratio = np.sqrt(eigenvalues[-1] / max(eigenvalues[0], 1e-8))
        ax.fill(closed[:, 0], closed[:, 1], color="tab:blue", alpha=0.65)
        ax.plot(closed[:, 0], closed[:, 1], "k-", lw=0.7)
        ax.set(
            title=(
                f"n={int(length)}, AR={aspect_ratio:.2f}, "
                f"CV={radial_cv:.2f}, C={circularity:.2f}"
            ),
            xlim=(-0.75, 0.75),
            ylim=(-0.75, 0.75), aspect="equal",
        )
        ax.axis("off")
    fig.suptitle(
        f"40 Fourier-smoothed training samples (area={OBJECT_TARGET_AREA})",
        fontsize=14,
    )
    display(fig)
    plt.close(fig)


def render_drag_comparison(
    parameters: torch.Tensor,
    lengths: torch.Tensor,
    true_drag: torch.Tensor,
    predicted_drag: torch.Tensor,
    adversarial_drag: torch.Tensor,
    step: int,
) -> None:
    parameters = parameters.detach().cpu()
    lengths = lengths.detach().cpu()
    true_drag = true_drag.detach().cpu()
    predicted_drag = predicted_drag.detach().cpu()
    adversarial_drag = adversarial_drag.detach().cpu()
    normalized_true_drag = normalize_drag(true_drag)
    normalized_predicted_drag = normalize_drag(predicted_drag)
    normalized_adversarial_drag = normalize_drag(adversarial_drag)
    fig, axes = plt.subplots(2, 5, figsize=(14, 6), constrained_layout=True)
    for (
        ax, param, length, target, prediction, attacked_prediction,
        normalized_target, normalized_prediction, normalized_attacked_prediction,
    ) in zip(
        axes.flat, parameters, lengths, true_drag, predicted_drag, adversarial_drag,
        normalized_true_drag, normalized_predicted_drag, normalized_adversarial_drag,
    ):
        vertices = parameters_to_vertices(param[:length]).numpy()
        closed = np.vstack((vertices, vertices[0]))
        ax.fill(closed[:, 0], closed[:, 1], color="tab:cyan", alpha=0.65)
        ax.plot(closed[:, 0], closed[:, 1], color="#16324f", lw=1.0)
        ax.annotate(
            "wind →", xy=(-1.0, 0.0), xytext=(-1.0, 0.0),
            fontsize=9, ha="right", va="center", color="tab:blue",
        )
        comparison_text = (
            f"LBM  $C_d$={target:.3f}, z={normalized_target:.2f}\n"
            f"model $C_d$={prediction:.3f}, z={normalized_prediction:.2f}"
        )
        if torch.isfinite(attacked_prediction):
            comparison_text += (
                f"\nPGD   $C_d$={attacked_prediction:.3f}, "
                f"z={normalized_attacked_prediction:.2f}"
            )
        ax.text(
            0.03, 0.97,
            comparison_text,
            transform=ax.transAxes, ha="left", va="top", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.88, edgecolor="none"),
        )
        ax.set(xlim=(-1.25, 1.25), ylim=(-1.25, 1.25), aspect="equal")
        ax.axis("off")
    fig.suptitle(f"Fresh validation shapes after training step {step}", fontsize=14)
    display(fig)
    plt.close(fig)


def render_clipped_ema_loss(
    history: dict[str, list[float]], current_step: int, current_ema: float
) -> None:
    """Winsorize EMA values in the top decile for a readable plot."""
    steps = list(history["step"])
    ema_values = list(history["ema_loss"])
    if not steps or steps[-1] != current_step:
        steps.append(current_step)
        ema_values.append(current_ema)

    ema_array = np.asarray(ema_values, dtype=np.float64)
    decile_edges = np.quantile(ema_array, np.linspace(0.0, 1.0, 11))
    top_decile_threshold = float(decile_edges[-2])
    clipped_ema = np.minimum(ema_array, top_decile_threshold)
    visible_min = float(clipped_ema.min())
    padding = max(
        1e-8,
        0.05 * max(abs(top_decile_threshold), top_decile_threshold - visible_min),
    )

    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(steps, clipped_ema, color="tab:purple", lw=2, marker="o", ms=3)
    ax.axhline(
        top_decile_threshold, color="tab:red", ls="--", lw=1,
        label="top-decile threshold",
    )
    ax.set_ylim(
        bottom=max(0.0, visible_min - padding),
        top=top_decile_threshold + padding,
    )
    ax.set(
        xlabel="optimizer step", ylabel="EMA loss",
        title=f"EMA training loss at debug step {current_step}",
    )
    ax.grid(alpha=0.25)
    ax.legend()
    display(fig)
    plt.close(fig)


def validate_and_render(
    model: nn.Module, flow_config: FlowConfig, training_config: TrainingConfig,
    count: int, step: int,
) -> tuple[float, float, float]:
    was_training = model.training
    model.eval()
    parameters, lengths, _ = sample_random_polylines(count, device=DEVICE)
    targets = estimate_drag_lbm(parameters, lengths, flow_config)
    with torch.inference_mode():
        normalized_predictions = model(parameters, lengths)
        predictions = denormalize_drag(normalized_predictions)
    if training_config.adversarial_enabled:
        attacked_parameters = pgd_adversarial_parameters(
            model, parameters, lengths, targets, training_config, random_start=True
        )
        with torch.inference_mode():
            normalized_adversarial_predictions = model(attacked_parameters, lengths)
            adversarial_predictions = denormalize_drag(
                normalized_adversarial_predictions
            )
        adversarial_mae = (adversarial_predictions - targets).abs().mean().item()
    else:
        adversarial_predictions = torch.full_like(predictions, torch.nan)
        adversarial_mae = float("nan")
    mae = (predictions - targets).abs().mean().item()
    relative_mae = ((predictions - targets).abs() / targets.clamp_min(1e-4)).mean().item()
    render_drag_comparison(
        parameters, lengths, targets, predictions, adversarial_predictions, step
    )
    model.train(was_training)
    return mae, relative_mae, adversarial_mae


def train_drag_model(
    model: nn.Module,
    flow_config: FlowConfig = FLOW,
    training_config: TrainingConfig = TrainingConfig(),
) -> dict[str, list[float]]:
    preview_parameters, preview_lengths, _ = sample_random_polylines(
        40, device=DEVICE
    )
    preview_areas = polyline_areas(preview_parameters, preview_lengths)
    assert torch.allclose(
        preview_areas, torch.full_like(preview_areas, OBJECT_TARGET_AREA), atol=2e-5
    )
    render_sampled_shapes(preview_parameters, preview_lengths)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=training_config.learning_rate,
        weight_decay=training_config.weight_decay,
    )
    history = {
        "step": [], "loss": [], "ema_loss": [],
        "clean_loss": [], "adversarial_loss": [],
        "validation_mae": [], "validation_relative_mae": [],
        "validation_adversarial_mae": [],
    }
    model.train()
    running_loss = 0.0
    ema_loss = None
    ema_decay = 0.95
    interval_start = time.perf_counter()

    progress = trange(1, training_config.steps + 1, desc="training drag model", unit="step")
    for step in progress:
        parameters, lengths, _ = sample_random_polylines(
            training_config.batch_size, device=DEVICE
        )
        targets = estimate_drag_lbm(parameters, lengths, flow_config)

        optimizer.zero_grad(set_to_none=True)
        if training_config.adversarial_enabled:
            attacked_parameters = pgd_adversarial_parameters(
                model, parameters, lengths, targets, training_config, random_start=True
            )
            optimizer.zero_grad(set_to_none=True)
        clean_predictions = model(parameters, lengths)
        clean_loss = drag_regression_loss(clean_predictions, targets)
        if training_config.adversarial_enabled:
            adversarial_predictions = model(attacked_parameters, lengths)
            adversarial_loss = drag_regression_loss(adversarial_predictions, targets)
            loss = (
                (1.0 - training_config.adversarial_weight) * clean_loss
                + training_config.adversarial_weight * adversarial_loss
            )
        else:
            adversarial_loss = torch.full_like(clean_loss, torch.nan)
            loss = clean_loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), training_config.gradient_clip)
        optimizer.step()
        current_loss = loss.item()
        running_loss += current_loss
        ema_loss = current_loss if ema_loss is None else ema_decay * ema_loss + (1.0 - ema_decay) * current_loss
        postfix = {
            "loss": f"{current_loss:.5f}", "ema_loss": f"{ema_loss:.5f}"
        }
        if training_config.adversarial_enabled:
            postfix["adv_loss"] = f"{adversarial_loss.item():.5f}"
        progress.set_postfix(**postfix)

        if step % 10 == 0:
            history["step"].append(step)
            history["loss"].append(running_loss / 10.0)
            history["ema_loss"].append(ema_loss)
            history["clean_loss"].append(clean_loss.item())
            history["adversarial_loss"].append(adversarial_loss.item())
            running_loss = 0.0

        if step % training_config.plot_every == 0:
            elapsed = time.perf_counter() - interval_start
            mae, relative_mae, adversarial_mae = validate_and_render(
                model, flow_config, training_config, training_config.validation_shapes, step
            )
            history["validation_mae"].append(mae)
            history["validation_relative_mae"].append(relative_mae)
            history["validation_adversarial_mae"].append(adversarial_mae)
            render_clipped_ema_loss(history, step, ema_loss)
            latest_loss = history["loss"][-1] if history["loss"] else loss.item()
            validation_message = (
                f"step {step:5d} | loss {latest_loss:.5f} | "
                f"EMA loss {ema_loss:.5f} | validation MAE {mae:.4f} | "
            )
            if training_config.adversarial_enabled:
                validation_message += f"PGD MAE {adversarial_mae:.4f} | "
            validation_message += (
                f"relative MAE {relative_mae:.2%} | "
                f"{elapsed:.1f}s / {training_config.plot_every} steps"
            )
            progress.write(
                validation_message
            )
            interval_start = time.perf_counter()

    return history

In [ ]:
# This cell performs online simulation and training. On CPU, begin with fewer
# steps or a smaller batch; on GPU the default configuration is recommended.
TRAINING = TrainingConfig()
history = train_drag_model(model, FLOW, TRAINING)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history["step"], history["loss"], color="tab:blue", alpha=0.45, label="10-step mean")
ax.plot(history["step"], history["ema_loss"], color="tab:blue", label="EMA (decay 0.95)")
ax.set(xlabel="optimizer step", ylabel="mean training loss", title="Online drag-regression training")
ax.grid(alpha=0.25)
ax.legend()
plt.show()